# NeRF 体渲染从零实现：展柜光带重建

## 面试问题

面试时我会先把 NeRF 分成两部分：MLP 把三维位置映射为体密度和颜色，体渲染再沿每条相机射线把离散采样转换成像素。位置编码用多频正余弦帮助 MLP 表达高频细节。每个采样点的 alpha 必须包含采样间距 delta，透射率是之前所有点未被吸收概率的累乘，权重等于透射率乘当前 alpha。训练通过像素重建误差把梯度穿过渲染公式传回 MLP。下面用十条可读相机射线比较平均颜色基线，手写位置编码、NeRF MLP 和 alpha compositing，并量化漏乘 delta 的错误。

## 真实案例

场景是一条悬浮在商品展柜中的彩色半透明光带。十台虚拟相机从不同横向位置沿 z 轴发射射线，目标 RGB 由已知解析辐射场离线渲染得到，同时保留期望深度。该数据只用于验证数学机制，不代表真实多视角重建质量。

本实验是用于解释机制的确定性小样本，所有指标均标记为“教学实验”，不能外推为线上收益。

In [1]:
import math  # 导入圆周率用于多频位置编码。
import torch  # 导入 PyTorch 以实现可导体渲染。
from torch import nn  # 导入神经网络基础层。
import torch.nn.functional as F  # 导入 softplus、激活和重建损失。
torch.manual_seed(48)  # 固定随机种子以复现实验输出。
torch.set_num_threads(1)  # 限制 CPU 线程以稳定小实验耗时。
ray_names = [f"相机-{index:02d}" for index in range(1, 11)]  # 定义十条可读相机射线编号。
ray_offsets = torch.linspace(-0.65, 0.65, len(ray_names))  # 设置十个横向相机位置。
ray_origins = torch.stack([ray_offsets, torch.zeros_like(ray_offsets), torch.full_like(ray_offsets, -1.0)], dim=1)  # 构造位于展柜前方的十个射线原点。
ray_directions = torch.tensor([[0.0, 0.0, 1.0]], dtype=torch.float32).repeat(len(ray_names), 1)  # 让所有相机沿 z 轴观察光带。
sample_depths = torch.linspace(0.0, 2.0, 32)  # 在每条射线上等距采样三十二个深度。
def volume_render(density, colors, depths, use_delta=True):  # 手写离散 NeRF alpha compositing 公式。
    intervals = depths[1:] - depths[:-1]  # 计算相邻采样点之间的真实距离。
    intervals = torch.cat([intervals, intervals[-1:]])  # 为最后采样点复用末段距离。
    effective_intervals = intervals if use_delta else torch.ones_like(intervals)  # 错误模式故意忽略采样间距。
    alpha = 1.0 - torch.exp(-density * effective_intervals.unsqueeze(0))  # 把体密度与路径长度转换为不透明度。
    survival = torch.cat([torch.ones_like(alpha[:, :1]), 1.0 - alpha + 1e-10], dim=1)  # 在累乘前补上初始全透射状态。
    transmittance = torch.cumprod(survival, dim=1)[:, :-1]  # 计算每个采样点之前光线仍存活的概率。
    weights = transmittance * alpha  # 计算每个采样点对最终像素的贡献权重。
    rendered_color = (weights.unsqueeze(-1) * colors).sum(dim=1)  # 沿射线加权汇总 RGB。
    opacity = weights.sum(dim=1)  # 汇总每条射线的总不透明度。
    expected_depth = (weights * depths.unsqueeze(0)).sum(dim=1) / opacity.clamp(min=1e-6)  # 计算有颜色贡献位置的期望深度。
    return rendered_color, weights, expected_depth, opacity  # 返回像素、采样权重、深度与不透明度。
def analytic_field(points):  # 定义用于产生监督像素的解析展柜光带。
    x_coordinate = points[..., 0]  # 读取每个采样点的横向坐标。
    z_coordinate = points[..., 2]  # 读取每个采样点的纵深坐标。
    density = 28.0 * torch.exp(-((x_coordinate / 0.48) ** 2 + ((z_coordinate - 0.72) / 0.18) ** 2))  # 用高斯体密度表示半透明光带。
    red = torch.sigmoid(-4.0 * x_coordinate)  # 让光带左侧偏红。
    green = 0.25 + 0.55 * torch.exp(-(x_coordinate / 0.35) ** 2)  # 让中间区域具有更强绿色。
    blue = torch.sigmoid(4.0 * x_coordinate)  # 让光带右侧偏蓝。
    colors = torch.stack([red, green, blue], dim=-1)  # 合并每个点的解析 RGB。
    return density, colors  # 返回解析密度和颜色供离线监督生成。
ray_points = ray_origins[:, None, :] + ray_directions[:, None, :] * sample_depths[None, :, None]  # 计算十条射线上的全部三维采样点。
target_density, target_sample_colors = analytic_field(ray_points)  # 查询解析场得到监督密度与采样颜色。
target_colors, target_weights, target_depths, target_opacity = volume_render(target_density, target_sample_colors, sample_depths)  # 用正确体渲染产生目标像素与深度。
print("射线      原点 x   目标 RGB                 期望深度  不透明度")  # 打印真实输入和监督结果表头。
for index, ray_name in enumerate(ray_names):  # 逐射线展示相机位置与目标像素。
    rgb = [round(value, 3) for value in target_colors[index].tolist()]  # 把当前目标 RGB 保留三位小数。
    print(f"{ray_name}  {ray_offsets[index]:>6.2f}   {rgb}   {target_depths[index]:.3f}     {target_opacity[index]:.3f}")  # 输出当前射线完整监督信息。
print(f"射线点张量={tuple(ray_points.shape)}，目标颜色张量={tuple(target_colors.shape)}")  # 汇总几何采样与像素张量形状。

射线      原点 x   目标 RGB                 期望深度  不透明度
相机-01   -0.65   [0.705, 0.203, 0.052]   1.670     0.758
相机-02   -0.51   [0.836, 0.301, 0.111]   1.626     0.946
相机-03   -0.36   [0.804, 0.437, 0.19]   1.583     0.994
相机-04   -0.22   [0.704, 0.624, 0.296]   1.554     0.999
相机-05   -0.07   [0.572, 0.777, 0.428]   1.540     1.000
相机-06    0.07   [0.428, 0.777, 0.572]   1.540     1.000
相机-07    0.22   [0.296, 0.624, 0.704]   1.554     0.999
相机-08    0.36   [0.19, 0.437, 0.804]   1.583     0.994
相机-09    0.51   [0.111, 0.301, 0.836]   1.626     0.946
相机-10    0.65   [0.052, 0.203, 0.705]   1.670     0.758
射线点张量=(10, 32, 3)，目标颜色张量=(10, 3)


## 基线：所有射线预测平均颜色

平均颜色基线完全忽略相机横向位置。它与 NeRF 使用相同十个目标像素和 RGB 均方误差。

In [2]:
mean_color = target_colors.mean(dim=0, keepdim=True)  # 计算十条监督射线的全局平均 RGB。
baseline_colors = mean_color.repeat(len(ray_names), 1)  # 对每条射线都输出同一个平均颜色。
baseline_sample_mse = ((baseline_colors - target_colors) ** 2).mean(dim=1)  # 计算逐射线平均颜色误差。
baseline_mse = baseline_sample_mse.mean().item()  # 汇总平均颜色基线 MSE。
print(f"平均颜色基线 RGB={[round(value, 3) for value in mean_color.squeeze(0).tolist()]}")  # 展示基线实际输出。
print("射线      基线 MSE")  # 打印逐射线基线误差表头。
for index, ray_name in enumerate(ray_names):  # 遍历十条射线展示基线误差分布。
    print(f"{ray_name}  {baseline_sample_mse[index]:.5f}")  # 输出当前射线的 RGB 均方误差。
print(f"平均颜色基线 MSE={baseline_mse:.5f}")  # 汇总同数据基线指标。

平均颜色基线 RGB=[0.47, 0.468, 0.47]
射线      基线 MSE
相机-01  0.10011
相机-02  0.09697
相机-03  0.06371
相机-04  0.03643
相机-05  0.03576
相机-06  0.03576
相机-07  0.03643
相机-08  0.06371
相机-09  0.09697
相机-10  0.10011
平均颜色基线 MSE=0.06660


## 手写核心：位置编码、密度颜色 MLP 与可导渲染

MLP 对每个三维采样点预测一个非负密度和 RGB；随后使用前面手写的 `volume_render` 得到每条射线像素。没有调用现成 NeRF 或渲染器。

In [3]:
def positional_encoding(points, frequency_count=4):  # 用多频正余弦扩展三维坐标。
    encoded_parts = [points]  # 保留原始坐标作为低频分量。
    for frequency_index in range(frequency_count):  # 逐频率加入正弦和余弦特征。
        frequency = (2.0 ** frequency_index) * math.pi  # 计算当前频带的角频率。
        encoded_parts.append(torch.sin(points * frequency))  # 加入当前频带正弦特征。
        encoded_parts.append(torch.cos(points * frequency))  # 加入当前频带余弦特征。
    return torch.cat(encoded_parts, dim=-1)  # 拼接原坐标和全部频带特征。
class TinyNeRF(nn.Module):  # 定义把三维位置映射到密度和颜色的教学版 NeRF。
    def __init__(self, encoded_dim):  # 根据位置编码维度创建共享 MLP 与两个输出头。
        super().__init__()  # 初始化父类以注册全部参数。
        self.layer_one = nn.Linear(encoded_dim, 48)  # 把多频位置特征映射到隐藏空间。
        self.layer_two = nn.Linear(48, 48)  # 继续提取空间辐射场表示。
        self.density_head = nn.Linear(48, 1)  # 为每个采样点输出未约束体密度。
        self.color_head = nn.Linear(48, 3)  # 为每个采样点输出 RGB logits。
    def forward(self, points):  # 对任意形状的三维点批次查询辐射场。
        encoded = positional_encoding(points)  # 计算三维坐标的多频位置编码。
        hidden_one = torch.relu(self.layer_one(encoded))  # 提取第一层非线性空间特征。
        hidden_two = torch.relu(self.layer_two(hidden_one))  # 提取第二层辐射场隐藏特征。
        density = F.softplus(self.density_head(hidden_two)).squeeze(-1)  # 用 softplus 保证体密度非负。
        colors = torch.sigmoid(self.color_head(hidden_two))  # 用 sigmoid 把预测颜色限制到零至一。
        return density, colors, hidden_two  # 返回密度、颜色和可观察隐藏表示。
encoded_dimension = positional_encoding(torch.zeros(1, 3)).shape[1]  # 动态计算四频位置编码后的维度。
nerf = TinyNeRF(encoded_dimension)  # 实例化手写辐射场 MLP。
print(nerf)  # 展示 NeRF 由基础线性层组成的真实结构。
print(f"位置编码维度={encoded_dimension}，可训练参数={sum(parameter.numel() for parameter in nerf.parameters())}")  # 输出编码与模型规模。

TinyNeRF(
  (layer_one): Linear(in_features=27, out_features=48, bias=True)
  (layer_two): Linear(in_features=48, out_features=48, bias=True)
  (density_head): Linear(in_features=48, out_features=1, bias=True)
  (color_head): Linear(in_features=48, out_features=3, bias=True)
)
位置编码维度=27，可训练参数=3892


In [4]:
optimizer = torch.optim.Adam(nerf.parameters(), lr=0.01)  # 创建优化器学习解析光带的像素监督。
loss_trace = []  # 保存像素重建损失轨迹。
first_gradient_norm = 0.0  # 预留首轮密度头梯度范数。
flattened_points = ray_points.reshape(-1, 3)  # 把射线与采样维合并供 MLP 批量查询。
for epoch in range(701):  # 在十条教学射线上执行七百零一次更新。
    optimizer.zero_grad()  # 清空上一轮累计梯度。
    predicted_density_flat, predicted_colors_flat, hidden_points = nerf(flattened_points)  # 查询全部三维采样点的密度与颜色。
    predicted_density = predicted_density_flat.view(len(ray_names), -1)  # 恢复射线与采样两维的密度张量。
    predicted_sample_colors = predicted_colors_flat.view(len(ray_names), -1, 3)  # 恢复射线、采样和 RGB 三维张量。
    rendered_colors, rendered_weights, rendered_depths, rendered_opacity = volume_render(predicted_density, predicted_sample_colors, sample_depths)  # 通过手写 alpha compositing 得到像素。
    loss = F.mse_loss(rendered_colors, target_colors)  # 计算十条射线的 RGB 重建均方误差。
    loss.backward()  # 把像素误差穿过体渲染公式传回 NeRF MLP。
    if epoch == 0:  # 首轮记录密度分支的真实梯度规模。
        first_gradient_norm = nerf.density_head.weight.grad.norm().item()  # 读取密度头首轮梯度范数。
    optimizer.step()  # 根据当前梯度更新辐射场参数。
    loss_trace.append(loss.item())  # 保存当前轮像素损失。
    if epoch in [0, 50, 250, 700]:  # 选择关键轮次输出训练轨迹。
        print(f"epoch={epoch:03d} RGB_MSE={loss.item():.6f} mean_opacity={rendered_opacity.mean().item():.3f}")  # 展示重建误差和透明度变化。
nerf.eval()  # 切换到评估模式生成稳定射线结果。
with torch.no_grad():  # 关闭评估阶段的梯度记录。
    predicted_density_flat, predicted_colors_flat, hidden_points = nerf(flattened_points)  # 重新查询全部采样点。
    predicted_density = predicted_density_flat.view(len(ray_names), -1)  # 恢复每条射线的密度序列。
    predicted_sample_colors = predicted_colors_flat.view(len(ray_names), -1, 3)  # 恢复每条射线的采样颜色。
    nerf_colors, nerf_weights, nerf_depths, nerf_opacity = volume_render(predicted_density, predicted_sample_colors, sample_depths)  # 渲染最终像素、权重和深度。
nerf_sample_mse = ((nerf_colors - target_colors) ** 2).mean(dim=1)  # 计算逐射线 NeRF RGB 误差。
nerf_mse = nerf_sample_mse.mean().item()  # 汇总同数据上的 NeRF MSE。
depth_mae = (nerf_depths - target_depths).abs().mean().item()  # 计算像素拟合之外的期望深度误差以暴露几何歧义。
peak_indices = nerf_weights.argmax(dim=1)  # 找出每条射线贡献最大的采样点编号。
print(f"首轮密度头梯度范数={first_gradient_norm:.6f}")  # 输出非零梯度证明渲染公式可导。
print(f"中间射线密度前八项={[round(value, 3) for value in predicted_density[5, :8].tolist()]}")  # 展示一条射线的采样密度中间量。
print(f"中间射线权重和={nerf_weights[5].sum().item():.4f}，峰值深度={sample_depths[peak_indices[5]].item():.3f}")  # 展示归一化前景权重与主要贡献位置。

epoch=000 RGB_MSE=0.075175 mean_opacity=0.743
epoch=050 RGB_MSE=0.000128 mean_opacity=0.893


epoch=250 RGB_MSE=0.000017 mean_opacity=0.887


epoch=700 RGB_MSE=0.000007 mean_opacity=0.885
首轮密度头梯度范数=0.017320
中间射线密度前八项=[2.046, 1.878, 1.862, 1.689, 1.569, 1.624, 1.731, 1.809]
中间射线权重和=0.9863，峰值深度=0.000


## 结果解读

逐射线输出目标 RGB、渲染 RGB、MSE、目标/预测深度和最大权重采样位置。这里 RGB 已拟合得很好，但所有射线方向平行，模型可以把颜色放到错误深度仍得到相同像素；深度 MAE 会诚实暴露这种单视角几何不可辨识性。这也是为什么真实 NeRF 必须依赖多视角，而不能只看训练像素 PSNR。

In [5]:
print("射线      目标 RGB              NeRF RGB              MSE      目标深度  预测深度  权重峰值t")  # 打印逐射线渲染结果表头。
for index, ray_name in enumerate(ray_names):  # 遍历十条射线展示完整重建证据。
    target_rgb = [round(value, 3) for value in target_colors[index].tolist()]  # 格式化当前目标 RGB。
    predicted_rgb = [round(value, 3) for value in nerf_colors[index].tolist()]  # 格式化当前 NeRF RGB。
    peak_depth = sample_depths[peak_indices[index]].item()  # 读取当前射线最大体渲染权重对应深度。
    print(f"{ray_name}  {target_rgb}  {predicted_rgb}  {nerf_sample_mse[index]:.6f}  {target_depths[index]:.3f}    {nerf_depths[index]:.3f}    {peak_depth:.3f}")  # 输出当前射线的像素与几何对照。
print(f"同数据 RGB MSE：平均颜色基线={baseline_mse:.5f}，手写 NeRF={nerf_mse:.6f}")  # 汇总基线与 NeRF 的同口径误差。
print(f"几何诚实检查：期望深度 MAE={depth_mae:.3f}，说明平行射线仅凭 RGB 不能唯一恢复深度。")  # 明确区分像素拟合成功与几何恢复失败。

射线      目标 RGB              NeRF RGB              MSE      目标深度  预测深度  权重峰值t
相机-01  [0.705, 0.203, 0.052]  [0.708, 0.204, 0.052]  0.000002  1.670    0.806    0.000
相机-02  [0.836, 0.301, 0.111]  [0.838, 0.303, 0.109]  0.000004  1.626    0.711    0.000
相机-03  [0.804, 0.437, 0.19]  [0.806, 0.439, 0.188]  0.000004  1.583    0.673    0.000
相机-04  [0.704, 0.624, 0.296]  [0.706, 0.625, 0.294]  0.000004  1.554    0.634    0.000
相机-05  [0.572, 0.777, 0.428]  [0.576, 0.778, 0.424]  0.000010  1.540    0.485    0.000
相机-06  [0.428, 0.777, 0.572]  [0.432, 0.778, 0.568]  0.000009  1.540    0.464    0.000
相机-07  [0.296, 0.624, 0.704]  [0.298, 0.626, 0.702]  0.000004  1.554    0.613    0.000
相机-08  [0.19, 0.437, 0.804]  [0.192, 0.438, 0.802]  0.000002  1.583    0.672    0.000
相机-09  [0.111, 0.301, 0.836]  [0.112, 0.302, 0.835]  0.000001  1.626    0.719    0.000
相机-10  [0.052, 0.203, 0.705]  [0.053, 0.203, 0.705]  0.000000  1.670    0.799    0.000
同数据 RGB MSE：平均颜色基线=0.06660，手写 NeRF=0.000004
几何诚实检查：期望深度

## 失败案例：alpha 计算漏乘采样间距 delta

对长度约 2、恒定密度 0.5 的介质，理论不透明度约为 `1-exp(-1)`。若直接写 `1-exp(-sigma)`，每个采样点都被当成长度 1，32 个点就几乎完全不透明；修复后改变采样数仍接近同一物理结果。

In [6]:
demo_depths_32 = torch.linspace(0.0, 2.0, 32)  # 创建三十二点的恒定介质采样。
demo_depths_64 = torch.linspace(0.0, 2.0, 64)  # 创建六十四点的同一物理介质采样。
demo_density_32 = torch.full((1, 32), 0.5)  # 设置三十二点采样的恒定体密度。
demo_density_64 = torch.full((1, 64), 0.5)  # 设置六十四点采样的恒定体密度。
demo_colors_32 = torch.ones(1, 32, 3)  # 设置三十二点采样的白色辐射。
demo_colors_64 = torch.ones(1, 64, 3)  # 设置六十四点采样的白色辐射。
correct_color_32, correct_weights_32, correct_depth_32, correct_opacity_32 = volume_render(demo_density_32, demo_colors_32, demo_depths_32, use_delta=True)  # 使用真实间距渲染三十二点介质。
correct_color_64, correct_weights_64, correct_depth_64, correct_opacity_64 = volume_render(demo_density_64, demo_colors_64, demo_depths_64, use_delta=True)  # 使用真实间距渲染六十四点介质。
wrong_color_32, wrong_weights_32, wrong_depth_32, wrong_opacity_32 = volume_render(demo_density_32, demo_colors_32, demo_depths_32, use_delta=False)  # 复现漏乘 delta 的错误三十二点渲染。
theoretical_opacity = 1.0 - math.exp(-0.5 * 2.0)  # 计算连续恒定介质的理论不透明度。
correct_error = abs(correct_opacity_32.item() - theoretical_opacity)  # 计算正确离散渲染到理论值的误差。
wrong_error = abs(wrong_opacity_32.item() - theoretical_opacity)  # 计算错误渲染到理论值的误差。
sampling_gap = abs(correct_opacity_32.item() - correct_opacity_64.item())  # 测量正确公式改变采样数后的差异。
print(f"理论不透明度={theoretical_opacity:.4f}")  # 展示连续体渲染的参考值。
print(f"错误漏乘 delta：32 点不透明度={wrong_opacity_32.item():.4f}")  # 展示错误公式造成近乎全不透明。
print(f"修复后：32 点={correct_opacity_32.item():.4f}，64 点={correct_opacity_64.item():.4f}，采样差={sampling_gap:.4f}")  # 展示正确公式接近物理值且对采样数稳定。
print("修复结论：alpha 必须使用每段真实 delta，非均匀采样时逐点 delta 更不能省略。")  # 给出体渲染实现的明确门禁。

理论不透明度=0.6321
错误漏乘 delta：32 点不透明度=1.0000
修复后：32 点=0.6438，64 点=0.6379，采样差=0.0059
修复结论：alpha 必须使用每段真实 delta，非均匀采样时逐点 delta 更不能省略。


## 生产差距

真实 NeRF 需要相机内外参、分层采样、视角方向、背景、空间加速结构和独立新视角评估。大场景还涉及 hash grid、occupancy grid、混合精度和显存分块。本例只拟合十条平行射线，不能证明几何恢复或自由视角合成能力。

## 最小回归测试

In [7]:
assert len(ray_names) >= 5  # 保证案例包含足够多的可读相机射线。
assert loss_trace[-1] < loss_trace[0]  # 保证像素损失经过真实训练下降。
assert first_gradient_norm > 0.0  # 保证梯度穿过体渲染到达密度头。
assert nerf_mse < baseline_mse  # 保证 NeRF 在同一 RGB 指标上优于平均颜色基线。
assert torch.isfinite(nerf_weights).all() and torch.isfinite(nerf_colors).all()  # 保证渲染权重与像素没有数值异常。
assert correct_error < wrong_error  # 保证乘 delta 的公式比错误公式更接近物理理论值。
assert sampling_gap < 0.02  # 保证正确公式在两种采样密度下近似一致。
print("回归测试通过：NeRF 训练、体渲染权重和 delta 修复均符合预期。")  # 输出集中断言的最终验收结论。

回归测试通过：NeRF 训练、体渲染权重和 delta 修复均符合预期。
